# 15 — FilteredIndex on Real Recipe Data

`FilteredIndex` wraps any index (here a `HelixIndex`) with a SQL metadata
pre-filter and role-based access control. Filters are applied *before*
ranking, so a selective filter still returns a full top-k. Filters can be
written as SQL `WHERE` clauses or LangChain-style dicts, and every result
below is asserted against the source data.

In [1]:
from __future__ import annotations

import json
import tempfile
import time
from pathlib import Path

import pandas as pd

from simlar import CallableEmbedder, FilteredIndex, FilterError, HelixIndex, load_from_directory

CSV_PATH = Path("/home/aramirez/tekdatum/pgvector_recipe_demo/data/recipes_sample.csv")
MODEL = "sentence-transformers/all-MiniLM-L6-v2"
K = 5

LIST_COLUMNS = {
    "cuisine_list": "cuisines",
    "course_list": "courses",
    "tastes": "tastes",
    "dietary_profile": "diets",
    "health_flags": "health_flags",
}

## Helpers

Metadata is numbers, categories and lists -- no booleans (dietary flags are
the `diets` list instead). The toy access model makes `hard` recipes
premium-only. `show` prints results and asserts each one satisfies `check(row)`.

In [2]:
def load_recipes(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    for src, dst in LIST_COLUMNS.items():
        df[dst] = df[src].apply(json.loads)
    df["prep_min"] = df["est_prep_time_min"]
    df["cook_min"] = df["est_cook_time_min"]
    df["total_min"] = df["prep_min"] + df["cook_min"]
    df = df.reset_index(drop=True)
    df["id"] = [f"r{i}" for i in range(len(df))]
    return df


def recipe_metadata(row) -> dict:
    """One recipe's filterable metadata: numbers, categories and lists -- no
    booleans (dietary flags are the `diets` list instead)."""
    return {
        # numeric
        "total_min": int(row.total_min),
        "prep_min": int(row.prep_min),
        "cook_min": int(row.cook_min),
        "healthiness_score": int(row.healthiness_score),
        "num_ingredients": int(row.num_ingredients),
        "num_steps": int(row.num_steps),
        # categorical
        "difficulty": row.difficulty,
        "category": row.category,
        "primary_taste": row.primary_taste,
        "cook_speed": row.cook_speed,
        "main_ingredient": row.main_ingredient,
        "health_level": row.health_level,
        # multi-valued
        "cuisines": list(row.cuisines),
        "courses": list(row.courses),
        "tastes": list(row.tastes),
        "diets": list(row.diets),
        "health_flags": list(row.health_flags),
    }


def recipe_roles(row) -> list[str]:
    """Toy access model: 'hard' recipes are premium-only, the rest are
    visible to both tiers."""
    return ["premium"] if row.difficulty == "hard" else ["free", "premium"]


def show(title: str, df: pd.DataFrame, results, check=None, extra=()) -> None:
    """Print results and assert each one satisfies `check(row)`."""
    print(f"\n── {title}")
    if not results:
        print("   (no results)")
    cols = ["difficulty", "total_min", "healthiness_score", *extra]
    for r in results:
        row = df.loc[int(r.id[1:])]
        info = ", ".join(f"{c}={_fmt(row[c])}" for c in cols)
        print(f"   {r.score:7.4f}  {row.recipe_title[:48]:<48}  {info}")
        if check is not None:
            assert check(row), f"filter violated by {r.id}: {row.recipe_title!r}"


def _fmt(value) -> str:
    return "/".join(value) if isinstance(value, list) else str(value)


def count(idx: FilteredIndex, filter=None, roles=None) -> int:
    return len(idx.candidates(filter, roles))

## Load the data and set up an embedder

The embedder turns recipe texts into vectors on `add()` and query text into a vector on `search()`.

In [3]:
df = load_recipes(CSV_PATH)
print(f"Loaded {len(df)} recipes from {CSV_PATH}")

from sentence_transformers import SentenceTransformer

embedder = CallableEmbedder(SentenceTransformer(MODEL).encode, normalize=True, batch_size=64)
texts = df["combined_text"].fillna("").tolist()

# Vector-only searches (as in LangChain's similarity_search_by_vector) take a
# query vector; hybrid searches take just the text.
embed = embedder.embed_query

Loaded 2026 recipes from /home/aramirez/tekdatum/pgvector_recipe_demo/data/recipes_sample.csv


/home/aramirez/anaconda3/envs/simlar/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18673.58it/s]

## Build the FilteredIndex

In [4]:
t0 = time.perf_counter()
idx = FilteredIndex(HelixIndex(embedder=embedder))  # text_k / vector_k tuned by the engine
idx.add(
    df["id"].tolist(),
    texts,
    metadata=[recipe_metadata(row) for row in df.itertuples()],
    roles=[recipe_roles(row) for row in df.itertuples()],
)
scalar_cols, multi_keys = idx.sql_filter.schema()
print(f"Embedded with {MODEL} and built FilteredIndex in {time.perf_counter() - t0:.1f}s "
      f"over {idx.size} recipes")
print(f"  speed_preference: {idx.speed_preference}  expected recall: {idx.expected_recall}")
print(f"  scalar columns: {sorted(scalar_cols)}")
print(f"  multi-valued:   {sorted(multi_keys)}")

Embedded with sentence-transformers/all-MiniLM-L6-v2 and built FilteredIndex in 1.3s over 2026 recipes
  speed_preference: balanced  expected recall: None
  scalar columns: ['category', 'cook_min', 'cook_speed', 'difficulty', 'health_level', 'healthiness_score', 'main_ingredient', 'num_ingredients', 'num_steps', 'prep_min', 'primary_taste', 'total_min']
  multi-valued:   ['courses', 'cuisines', 'diets', 'health_flags', 'tastes']


## 1. Vector-only search, numeric filter

The shape of LangChain's `similarity_search_by_vector(query_vector, filter=...)`.

In [5]:
q_text = "quick weeknight chicken dinner"
q = embed(q_text)
f = "total_min <= 30"
print(f"[1] vector-only, filter={f!r}  ({count(idx, f)} recipes qualify)")
show("unfiltered", df, idx.search(query_vector=q, k=K))
show(f"filter: {f}", df, idx.search(query_vector=q, k=K, filter=f),
     check=lambda r: r.total_min <= 30)

[1] vector-only, filter='total_min <= 30'  (180 recipes qualify)

── unfiltered
    0.3333  Rustic Italian Chicken                            difficulty=medium, total_min=49, healthiness_score=80
    0.2500  Southwest Chicken                                 difficulty=medium, total_min=61, healthiness_score=76
    0.2000  Chicken Satay                                     difficulty=medium, total_min=88, healthiness_score=58
    0.1667  Best Chicken Salad                                difficulty=medium, total_min=43, healthiness_score=82
    0.1429  Chicken Piccata with Fettuccine                   difficulty=medium, total_min=69, healthiness_score=70

── filter: total_min <= 30
    0.3333  Best Chicken Salad Ever                           difficulty=easy, total_min=30, healthiness_score=78
    0.2500  Skillet Chicken Bulgogi                           difficulty=medium, total_min=30, healthiness_score=70
    0.2000  California Club Chicken Wraps                     difficulty=easy, tot

## 2. Hybrid search, multi-valued + numeric

In [6]:
f = "'dessert' IN courses AND healthiness_score >= 70"
print(f"[2] hybrid 'chocolate cake', filter={f!r}  ({count(idx, f)} qualify)")
show(f"filter: {f}", df,
     idx.search(query_text="chocolate cake", k=K, filter=f),
     check=lambda r: "dessert" in r.courses and r.healthiness_score >= 70, extra=("courses",))

[2] hybrid 'chocolate cake', filter="'dessert' IN courses AND healthiness_score >= 70"  (48 qualify)



── filter: 'dessert' IN courses AND healthiness_score >= 70
    0.6667  100% Fruit "Cake"                                 difficulty=medium, total_min=60, healthiness_score=78, courses=dessert
    0.3429  Mini Christmas Cakes                              difficulty=hard, total_min=149, healthiness_score=70, courses=dessert/bread/main
    0.3333  Christmas Nut Cake                                difficulty=medium, total_min=59, healthiness_score=70, courses=dessert/snack/bread/main
    0.3111  Lemon Blueberry Coffee Cake                       difficulty=hard, total_min=172, healthiness_score=70, courses=dessert/drink/main
    0.2917  Mini Crab Cakes                                   difficulty=medium, total_min=43, healthiness_score=76, courses=drink/main/dessert


## 3. Categories, `IN`, `BETWEEN`

In [7]:
f = "difficulty IN ('easy', 'medium') AND num_ingredients BETWEEN 3 AND 8 AND cook_speed = 'fast'"
print(f"[3] hybrid 'pasta', filter={f!r}  ({count(idx, f)} qualify)")
show(f"filter: {f}", df,
     idx.search(query_text="pasta", k=K, filter=f),
     check=lambda r: r.difficulty in ("easy", "medium") and 3 <= r.num_ingredients <= 8
     and r.cook_speed == "fast", extra=("num_ingredients", "cook_speed"))

[3] hybrid 'pasta', filter="difficulty IN ('easy', 'medium') AND num_ingredients BETWEEN 3 AND 8 AND cook_speed = 'fast'"  (558 qualify)

── filter: difficulty IN ('easy', 'medium') AND num_ingredients BETWEEN 3 AND 8 AND cook_speed = 'fast'
    0.5833  No-Mayo Pasta Salad                               difficulty=easy, total_min=28, healthiness_score=66, num_ingredients=7, cook_speed=fast
    0.4000  Kahala's Macaroni Seafood Salad                   difficulty=medium, total_min=53, healthiness_score=68, num_ingredients=6, cook_speed=fast
    0.3611  Pepperoni Casserole                               difficulty=medium, total_min=63, healthiness_score=50, num_ingredients=7, cook_speed=fast
    0.3361  Spaghetti with Garlic and Basil                   difficulty=easy, total_min=29, healthiness_score=70, num_ingredients=7, cook_speed=fast
    0.3095  Pesto Pasta with Chicken                          difficulty=medium, total_min=58, healthiness_score=74, num_ingredients=7, cook_speed=fast


## 4. Several list columns, `OR`, `NOT`

In [8]:
f = ("'vegan' IN diets AND ('spicy' IN tastes OR primary_taste = 'savory') "
     "AND NOT 'fried' IN health_flags")
q_text = "hearty stew"
q = embed(q_text)
print(f"[4] vector 'hearty stew', filter={f!r}  ({count(idx, f)} qualify)")
show(f"filter: {f}", df, idx.search(query_vector=q, k=K, filter=f),
     check=lambda r: "vegan" in r.diets and ("spicy" in r.tastes or r.primary_taste == "savory")
     and "fried" not in r.health_flags, extra=("diets", "tastes"))

[4] vector 'hearty stew', filter="'vegan' IN diets AND ('spicy' IN tastes OR primary_taste = 'savory') AND NOT 'fried' IN health_flags"  (55 qualify)

── filter: 'vegan' IN diets AND ('spicy' IN tastes OR primary_taste = 'savory') AND NOT 'fried' IN health_flags
    0.3333  Mushroom Lentil Barley Stew                       difficulty=medium, total_min=51, healthiness_score=90, diets=vegan/nut_free/halal/kosher, tastes=savory
    0.2500  Original Old Bay Coleslaw                         difficulty=medium, total_min=31, healthiness_score=76, diets=vegan/gluten_free/nut_free/halal/kosher, tastes=sweet/spicy/sour/savory
    0.2000  Slow Cooker Sweet Potato Chili                    difficulty=medium, total_min=49, healthiness_score=80, diets=vegan/gluten_free/nut_free/halal/kosher, tastes=sweet/spicy/savory
    0.1667  Cranberry Salsa                                   difficulty=easy, total_min=14, healthiness_score=76, diets=vegan/gluten_free/nut_free/halal/kosher, tastes=sweet/spicy/sour/

## 5. `LIKE`, and the same filter as a `MetadataFilter` chain

SQLite `LIKE` is ASCII case-insensitive. Strings and dicts are translated into a `MetadataFilter`, so a chain built by hand selects the same rows.

In [9]:
f = "category LIKE '%Bread%' AND health_level NOT LIKE 'unhealthy' AND prep_min <= 20"
sf = idx.sql_filter
chain = sf.where_like("category", "%Bread%").where("prep_min", "<=", 20) & ~sf.where_like(
    "health_level", "unhealthy"
)
assert idx.candidates(chain).tolist() == idx.candidates(f).tolist()
print(f"[5] hybrid 'banana bread', filter={f!r}  ({count(idx, f)} qualify)")
show(f"filter: {f}", df,
     idx.search(query_text="banana bread", k=K, filter=chain),
     check=lambda r: "bread" in r.category.lower() and r.health_level.lower() != "unhealthy"
     and r.prep_min <= 20, extra=("category", "prep_min", "health_level"))

[5] hybrid 'banana bread', filter="category LIKE '%Bread%' AND health_level NOT LIKE 'unhealthy' AND prep_min <= 20"  (46 qualify)

── filter: category LIKE '%Bread%' AND health_level NOT LIKE 'unhealthy' AND prep_min <= 20
    0.6667  Nutty Banana Bread                                difficulty=medium, total_min=48, healthiness_score=64, category=Banana Breads, prep_min=15, health_level=moderate
    0.4500  Banana Loaf                                       difficulty=medium, total_min=35, healthiness_score=58, category=Breads, prep_min=14, health_level=moderate
    0.4167  Brown Sugar Banana Nut Bread I                    difficulty=medium, total_min=40, healthiness_score=58, category=Banana Breads, prep_min=16, health_level=moderate
    0.3250  Hawaiian Banana Nut Bread                         difficulty=medium, total_min=39, healthiness_score=68, category=Breads, prep_min=18, health_level=moderate
    0.2917  Einkorn Banana Bread                              difficulty=medium, total

## 6. The same filter as a LangChain-style dict

In [10]:
d = {"$and": [
    {"courses": {"$contains": "soup"}},
    {"cuisines": {"$in": ["asian", "korean", "thai"]}},
    {"total_min": {"$lte": 60}},
]}
s = "'soup' IN courses AND cuisines IN ('asian', 'korean', 'thai') AND total_min <= 60"
assert idx.candidates(d).tolist() == idx.candidates(s).tolist()
print(f"[6] dict filter == SQL filter ({count(idx, d)} qualify)")
show("dict: soup, asian/korean/thai, <= 60 min", df,
     idx.search(query_vector=embed("noodle soup"), k=K, filter=d),
     check=lambda r: "soup" in r.courses and bool({"asian", "korean", "thai"} & set(r.cuisines))
     and r.total_min <= 60, extra=("cuisines",))

[6] dict filter == SQL filter (18 qualify)

── dict: soup, asian/korean/thai, <= 60 min
    0.3333  Wonton Soup                                       difficulty=medium, total_min=52, healthiness_score=62, cuisines=asian/chinese/european/filipino/french/japanese/korean
    0.2500  Easy Stroganoff                                   difficulty=medium, total_min=46, healthiness_score=62, cuisines=asian/european/french/korean/russian
    0.2000  Creamy Garlic Pasta                               difficulty=medium, total_min=53, healthiness_score=58, cuisines=asian/european/french/greek/italian/korean/mediterranean/middle eastern region
    0.1667  Creamy Garlic Pasta                               difficulty=medium, total_min=53, healthiness_score=58, cuisines=asian/european/french/greek/italian/korean/mediterranean/middle eastern region
    0.1429  Simple Strawberry Soup                            difficulty=easy, total_min=13, healthiness_score=76, cuisines=asian/european/greek/indian/medite

## 7. Roles

A filter can't widen access: the role check is appended by `FilteredIndex`.

In [11]:
q = embed("slow braised beef")
print(f"[7] roles: free sees {count(idx, roles=['free'])}, "
      f"premium sees {count(idx, roles=['premium'])} of {idx.size}")
show("roles=['free']  (no 'hard' recipes)", df, idx.search(query_vector=q, k=K, roles=["free"]),
     check=lambda r: r.difficulty != "hard")
show("roles=['premium']", df, idx.search(query_vector=q, k=K, roles=["premium"]))
show("roles=['free'], filter='1=1 OR 1=1'", df,
     idx.search(query_vector=q, k=K, roles=["free"], filter="1=1 OR 1=1"),
     check=lambda r: r.difficulty != "hard")

[7] roles: free sees 1289, premium sees 2026 of 2026

── roles=['free']  (no 'hard' recipes)
    0.3333  Slow Cooker Beef Tips with Mushroom Gravy         difficulty=medium, total_min=72, healthiness_score=86
    0.2500  Slow Cooker Beef Tips with Mushroom Gravy         difficulty=medium, total_min=72, healthiness_score=86
    0.2000  Slow Cooker Corned Beef Reuben Sandwiches         difficulty=medium, total_min=72, healthiness_score=78
    0.1667  Slow-Cooker Corned Beef and Cabbage               difficulty=medium, total_min=66, healthiness_score=78
    0.1429  BBQ Beef Brisket Sandwiches                       difficulty=medium, total_min=68, healthiness_score=76

── roles=['premium']
    0.3333  Filet with a Merlot Sauce                         difficulty=hard, total_min=96, healthiness_score=70
    0.2500  Slow Cooker Beef Tips with Mushroom Gravy         difficulty=medium, total_min=72, healthiness_score=86
    0.2000  Slow Cooker Beef Tips with Mushroom Gravy         difficulty=me

## 8. Pre-filter vs post-filter

A selective filter applied *after* an unrestricted top-k under-returns;
`FilteredIndex` applies it before ranking.

In [12]:
f = "'korean' IN cuisines AND 'breakfast' IN courses AND total_min <= 45"
q = embed("pancakes")
allowed = {f"r{p}" for p in idx.candidates(f)}
post = [r for r in idx.search(query_vector=q, k=K) if r.id in allowed]
pre = idx.search(query_vector=q, k=K, filter=f)
print(f"[8] {f!r}: {len(allowed)} qualify; "
      f"post-filtering the unfiltered top-{K} keeps {len(post)}, pre-filtering returns {len(pre)}")
assert len(pre) == min(K, len(allowed))

[8] "'korean' IN cuisines AND 'breakfast' IN courses AND total_min <= 45": 2 qualify; post-filtering the unfiltered top-5 keeps 0, pre-filtering returns 2


## 9. Rejected filters

In [13]:
print("[9] rejected filters")
for bad in [
    "total_min <= 30; DROP TABLE docs",
    "total_min IN (SELECT position FROM doc_access)",
    "load_extension('evil.so')",
    "calories < 500",                       # unknown column
    "healthiness_score > 50 AND tastes > 'a'",  # range on a list column
]:
    try:
        idx.search(query_vector=q, k=K, filter=bad)
        raise AssertionError(f"accepted: {bad}")
    except FilterError as e:
        print(f"   {bad!r:<52} -> {str(e)[:70]}")

[9] rejected filters
   'total_min <= 30; DROP TABLE docs'                   -> Could not parse filter 'total_min <= 30; DROP TABLE docs': Failed to p
   'total_min IN (SELECT position FROM doc_access)'     -> Subqueries are not allowed in a filter.
   "load_extension('evil.so')"                          -> Function 'load_extension' is not allowed in a filter.
   'calories < 500'                                     -> Unknown column 'calories' in filter.
   "healthiness_score > 50 AND tastes > 'a'"            -> Range comparison '>' is not supported on multi-valued column 'tastes':


## 10. Metadata updates + persistence

In [14]:
first = idx.search(query_vector=embed("pumpkin"), k=1)[0].id
idx.update_metadata([first], [{"healthiness_score": 999}])
assert [r.id for r in idx.search(query_vector=q, k=K, filter="healthiness_score = 999")] == [first]
with tempfile.TemporaryDirectory() as tmp:
    path = str(Path(tmp) / "recipes")
    idx.save(path)
    loaded = load_from_directory(path, embedder=embedder)  # embedders aren't saved
    f = "'dessert' IN courses AND total_min <= 30"
    a = [r.id for r in idx.search(query_vector=q, k=K, filter=f, roles=["free"])]
    b = [r.id for r in loaded.search(query_vector=q, k=K, filter=f, roles=["free"])]
    assert type(loaded) is FilteredIndex and a == b
print(f"[10] update_metadata + save/load round trip OK ({type(loaded.inner).__name__} inner)")

print("\nAll filtered results verified against the source data.")

[10] update_metadata + save/load round trip OK (HelixIndex inner)

All filtered results verified against the source data.
